In [ ]:
import pandas as pd
import numpy as np
from modelscope.msdatasets import MsDataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

/Users/jnkim/Desktop/coding stuff/phonefraudnlp/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Download & Explore Dataset

In [ ]:
print("Loading TeleAntiFraud-28k dataset...")
# For full dataset, use: ds = MsDataset.load('JimmyMa99/TeleAntiFraud-28k')
# For now, loading sample data created locally

df = pd.read_csv('raw_data.csv')
print(f"Dataset loaded! Shape: {df.shape}")

This may take a few minutes...



NameError: name 'MsDataset' is not defined

In [ ]:
print("\n=== DATASET SCHEMA ===")
print(f"Columns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nFirst row:")
for col in df.columns:
    value = df.iloc[0][col]
    if isinstance(value, str) and len(value) > 150:
        print(f"{col}: {value[:150]}...")
    else:
        print(f"{col}: {value}")

In [ ]:
# Identify key columns
text_cols = [col for col in df.columns if 'text' in col.lower() or 'transcript' in col.lower()]
label_cols = [col for col in df.columns if 'label' in col.lower() or 'fraud' in col.lower()]

text_col = text_cols[0] if text_cols else df.columns[0]
label_col = label_cols[0] if label_cols else None

print(f"Text column: {text_col}")
print(f"Label column: {label_col}")

print(f"\nTotal samples: {len(df):,}")
if label_col:
    print(f"\nLabel Distribution:")
    print(df[label_col].value_counts())
    print(f"\nPercentages:")
    print((df[label_col].value_counts() / len(df) * 100).round(2))

## Step 2: Extract Fraud Detection Features

class FraudDetectionFeatures:
    """
    Extract fraud detection features based on psychological manipulation cues
    """
    
    def __init__(self):
        self.urgency_words = ['immediately', 'urgent', 'now', 'expires', 'right away', 
                              'do not hang up', 'press 1 now', 'transfer today', 'act now',
                              'within 24 hours', 'asap', 'hurry', 'quickly']
        
        self.authority_terms = ['irs', 'bank', 'agent', 'fraud department', 'officer', 
                               'federal', 'government', 'police', 'fbi', 'official']
        
        self.threat_words = ['arrest', 'suspended', 'freeze', 'block', 'lawsuit', 
                            'fine', 'penalty', 'criminal', 'prosecution', 'charges']
        
        self.verification_patterns = ['ssn', 'social security', 'password', 'pin', 'account number',
                                     'credit card', 'confirm', 'verify', 'submit']
    
    def extract_urgency_score(self, text):
        if pd.isna(text):
            return 0
        text_lower = text.lower()
        return sum(1 for word in self.urgency_words if word in text_lower)
    
    def extract_authority_score(self, text):
        if pd.isna(text):
            return 0
        text_lower = text.lower()
        return sum(1 for term in self.authority_terms if term in text_lower)
    
    def extract_threat_score(self, text):
        if pd.isna(text):
            return 0
        text_lower = text.lower()
        return sum(1 for word in self.threat_words if word in text_lower)
    
    def extract_verification_requests(self, text):
        if pd.isna(text):
            return 0
        text_lower = text.lower()
        return sum(1 for pattern in self.verification_patterns if pattern in text_lower)
    
    def extract_all_features(self, df, text_column='text'):
        features_df = pd.DataFrame()
        features_df['urgency_score'] = df[text_column].apply(self.extract_urgency_score)
        features_df['authority_score'] = df[text_column].apply(self.extract_authority_score)
        features_df['threat_score'] = df[text_column].apply(self.extract_threat_score)
        features_df['verification_score'] = df[text_column].apply(self.extract_verification_requests)
        return features_df

print("Extracting custom fraud detection features...")
extractor = FraudDetectionFeatures()
custom_features = extractor.extract_all_features(df, text_col)

print(f"\nCustom Feature Statistics:")
print(custom_features.describe())

# Visualize custom features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
feature_cols = ['urgency_score', 'authority_score', 'threat_score', 'verification_score']

for idx, col in enumerate(feature_cols):
    ax = axes[idx // 2, idx % 2]
    custom_features[col].hist(bins=50, ax=ax, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(f'{col} Distribution')
    ax.set_ylabel('Frequency')
    ax.set_xlabel('Score')

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved feature_distributions.png")

# Combine features with original data
df_with_features = pd.concat([df, custom_features], axis=1)
print(f"\nDataset with features shape: {df_with_features.shape}")
print(f"New features added: {list(custom_features.columns)}")

## Step 3: Prepare Data & Train Model

# Remove NaN values
df_clean = df_with_features[df_with_features[text_col].notna()].copy()
if label_col:
    df_clean = df_clean[df_clean[label_col].notna()]

print(f"Samples after cleaning: {len(df_clean):,}")

# Prepare X and y
X_text = df_clean[text_col]
X_custom = df_clean[feature_cols].fillna(0)

if label_col:
    y = df_clean[label_col]
    # Convert to binary if needed
    if y.dtype == 'object':
        y = (y.astype(str).str.lower() != 'normal').astype(int)
    print(f"\nLabel distribution:")
    print(f"Class 0: {(y == 0).sum()} ({(y == 0).sum() / len(y) * 100:.1f}%)")
    print(f"Class 1: {(y == 1).sum()} ({(y == 1).sum() / len(y) * 100:.1f}%)")
else:
    print("Warning: No label column found")

# TF-IDF vectorization
print("Vectorizing text with TF-IDF...")
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english', ngram_range=(1, 2), min_df=2, max_df=0.9)
X_tfidf = vectorizer.fit_transform(X_text)

print(f"TF-IDF shape: {X_tfidf.shape}")
print(f"Custom features shape: {X_custom.shape}")

# Combine TF-IDF + custom features
X_combined = np.hstack([X_tfidf.toarray(), X_custom.values])
print(f"Combined features shape: {X_combined.shape}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Test samples: {X_test.shape[0]:,}")
print(f"\nTraining set: {(y_train == 0).sum()} normal, {(y_train == 1).sum()} fraud")
print(f"Test set: {(y_test == 0).sum()} normal, {(y_test == 1).sum()} fraud")

# Train Random Forest model
print("\nTraining Random Forest classifier...")
model = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"\n=== MODEL PERFORMANCE ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Fraud']))

In [ ]:
print("\n" + "="*70)
print("TELEANTIFRAUD-28K ANALYSIS SUMMARY")
print("="*70)
print(f"\nDataset:")
print(f"  Total samples: {len(df):,}")
print(f"  After cleaning: {len(df_clean):,}")
if label_col:
    print(f"  Normal calls: {(y == 0).sum():,}")
    print(f"  Fraud calls: {(y == 1).sum():,}")

print(f"\nFeatures:")
print(f"  TF-IDF features: {X_tfidf.shape[1]:,}")
print(f"  Custom features: {len(feature_cols)}")
print(f"    - Urgency Score")
print(f"    - Authority Score")
print(f"    - Threat Score")
print(f"    - Verification Score")
print(f"  Total features: {X_combined.shape[1]:,}")

print(f"\nModel Performance (Random Forest):")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  ROC-AUC: {roc_auc:.4f}")

print(f"\nFiles saved:")
print(f"  - feature_distributions.png")
print(f"  - confusion_matrix.png")
print(f"  - roc_curve.png")
print(f"  - feature_importance.png")
print("="*70)

## Step 4: Evaluation & Visualization

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(10, 8))
top_features = importance_df.head(20)
colors = ['#1f77b4' if feat in feature_cols else '#ff7f0e' for feat in top_features['Feature']]
ax.barh(range(len(top_features)), top_features['Importance'], color=colors)
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['Feature'])
ax.set_xlabel('Importance')
ax.set_title('Top 20 Most Important Features - TeleAntiFraud-28k')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
feature_importance = model.feature_importances_
all_feature_names = list(vectorizer.get_feature_names_out()) + feature_cols

importance_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Check custom features in top 20
custom_in_top20 = importance_df.head(20)[importance_df.head(20)['Feature'].isin(feature_cols)]
if len(custom_in_top20) > 0:
    print(f"\nCustom Features in Top 20:")
    print(custom_in_top20.to_string(index=False))

In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve - TeleAntiFraud-28k')
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Fraud'],
            yticklabels=['Normal', 'Fraud'], ax=ax)
ax.set_title('Confusion Matrix - TeleAntiFraud-28k')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")